# Graded Lab: Agentic Workflows

Build a multi-agent research workflow that plans, researches with tools, writes, and edits a short report.

## Setup

In [ ]:
# Standard-library imports.
from datetime import datetime
import ast
import json
import re

# Third-party and local imports.
from dotenv import load_dotenv
from IPython.display import Markdown, display
from aisuite import Client
import research_tools

# Load the API keys from .env and create the shared LLM client.
load_dotenv()
CLIENT = Client()

## Exercise 1: Planning agent

In [ ]:
def planner_agent(topic: str, model: str = 'openai:o4-mini') -> list[str]:
    # Create a short, executable plan for the research workflow.
    user_prompt = f'''
You are a planning agent responsible for organizing a research workflow with multiple intelligent agents.

Available agents: a research agent with arXiv, web, and Wikipedia tools; a writer agent; and an editor agent.
Return ONLY a valid Python list of atomic research-related step strings. Do not include explanations, setup steps,
or irrelevant tasks. The final step must generate a Markdown document containing the complete research report.

Topic: {topic}
'''
    messages = [{'role': 'user', 'content': user_prompt}]

    response = CLIENT.chat.completions.create(
        model=model,
        messages=messages,
        temperature=1,
    )

    steps = ast.literal_eval(response.choices[0].message.content.strip())
    if not isinstance(steps, list) or not all(isinstance(step, str) for step in steps):
        raise ValueError('The planner must return a Python list of strings.')
    return steps

## Exercise 2: Research agent

In [ ]:
def research_agent(task: str, model: str = 'openai:gpt-4o', return_messages: bool = False):
    # Use the available search tools to gather current and academic evidence.
    print('==================================')
    print('🔍 Research Agent')
    print('==================================')

    current_time = datetime.now().strftime('%Y-%m-%d')
    prompt = f'''
You are a research assistant. Use arxiv_search_tool for academic papers, tavily_search_tool for web sources,
and wikipedia_search_tool for concise background information. Cite source URLs and distinguish evidence from inference.

Current date: {current_time}
Task: {task}
'''
    messages = [{'role': 'user', 'content': prompt}]
    tools = [
        research_tools.arxiv_search_tool,
        research_tools.tavily_search_tool,
        research_tools.wikipedia_search_tool,
    ]

    response = CLIENT.chat.completions.create(
        model=model,
        messages=messages,
        tools=tools,
        tool_choice='auto',
        max_turns=6,
    )

    content = response.choices[0].message.content
    print('✅ Output:\n', content)
    return (content, messages) if return_messages else content

## Exercise 3: Writer agent

In [ ]:
def writer_agent(task: str, model: str = 'openai:gpt-4o') -> str:
    # Draft, expand, or summarize research content in a coherent academic style.
    print('==================================')
    print('✍️ Writer Agent')
    print('==================================')

    system_prompt = (
        'You are a writer agent specialized in clear, well-structured academic and technical writing. '
        'Draft sections, expand outlines into prose, and summarize complex material accurately.'
    )
    messages = [
        {'role': 'system', 'content': system_prompt},
        {'role': 'user', 'content': task},
    ]
    response = CLIENT.chat.completions.create(model=model, messages=messages, temperature=1.0)
    return response.choices[0].message.content

## Exercise 4: Editor agent

In [ ]:
def editor_agent(task: str, model: str = 'openai:gpt-4o') -> str:
    # Critique or revise a draft while preserving its supported meaning.
    print('==================================')
    print('🧠 Editor Agent')
    print('==================================')

    system_prompt = (
        'You are an editor agent. Reflect on strengths and weaknesses, critique clarity and argumentation, '
        'and revise drafts to improve coherence, style, and readability while preserving the original meaning.'
    )
    messages = [
        {'role': 'system', 'content': system_prompt},
        {'role': 'user', 'content': task},
    ]
    response = CLIENT.chat.completions.create(model=model, messages=messages, temperature=0.7)
    return response.choices[0].message.content

## Executor agent

In [ ]:
# The executor routes each plan step to the appropriate specialized agent.
agent_registry = {
    'research_agent': research_agent,
    'writer_agent': writer_agent,
    'editor_agent': editor_agent,
}


def clean_json_block(raw: str) -> str:
    # Remove optional Markdown fences before parsing the executor's JSON decision.
    raw = raw.strip()
    if raw.startswith('```'):
        raw = re.sub(r'^```(?:json)?\n?', '', raw)
        raw = re.sub(r'\n?```$', '', raw)
    return raw.strip()


def executor_agent(topic: str, model: str = 'openai:gpt-4o', limit_steps: bool = True):
    # Execute up to four plan steps, carrying previous results forward as context.
    plan_steps = planner_agent(topic)
    if limit_steps:
        plan_steps = plan_steps[:4]

    history = []
    for step in plan_steps:
        decision_prompt = f'''
You are an execution manager for a multi-agent research team.
Return only valid JSON with an agent key—one of research_agent, writer_agent, editor_agent—and a task key.

Instruction: {step}
'''
        decision = CLIENT.chat.completions.create(
            model=model,
            messages=[{'role': 'user', 'content': decision_prompt}],
            temperature=0,
        )
        agent_info = json.loads(clean_json_block(decision.choices[0].message.content))
        agent_name, task = agent_info['agent'], agent_info['task']

        context = '\n'.join(
            f'Step {index + 1} executed by {agent}:\n{result}'
            for index, (_, agent, result) in enumerate(history)
        )
        enriched_task = f'You are {agent_name}.\n\nPrevious context:\n{context}\n\nYour next task:\n{task}'
        print(f'\n🛠️ Executing with agent: `{agent_name}` on task: {task}')

        if agent_name not in agent_registry:
            raise ValueError(f'Unknown agent selected: {agent_name}')
        output = agent_registry[agent_name](enriched_task)
        history.append((step, agent_name, output))
        print(f'✅ Output:\n{output}')

    return history

## Run the workflow

In [ ]:
# Run after adding the API keys to .env. Set limit_steps=False only for a longer run.
executor_history = executor_agent('The ensemble Kalman filter for time series forecasting', limit_steps=True)
display(Markdown(executor_history[-1][-1].strip('`')))